# Spaceship Titanic — Machine Learning Analysis

## 標準的なMLワークフロー
1. データ読み込み・概観
2. 欠損値の確認
3. ターゲット変数の分布
4. 数値変数の分布・可視化
5. カテゴリ変数の可視化
6. 相関分析
7. 特徴量エンジニアリング
8. ベースラインモデル
9. 特徴量重要度の確認

## 0. ライブラリのインポート

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

print('ライブラリ読み込み完了')

## 1. データ読み込み・概観

In [ ]:
train = pd.read_csv('spaceship-titanic/train.csv')
test  = pd.read_csv('spaceship-titanic/test.csv')

print(f'train: {train.shape}')
print(f'test : {test.shape}')

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

## 2. 欠損値の確認

In [ ]:
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)

missing_df = pd.DataFrame({
    '欠損数': missing,
    '欠損率(%)': missing_pct
}).query('欠損数 > 0').sort_values('欠損率(%)', ascending=False)

print(missing_df)

missing_df['欠損率(%)'].plot(kind='bar', color='salmon')
plt.title('欠損値の割合 (%)')
plt.ylabel('欠損率 (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. ターゲット変数の分布

In [ ]:
target_counts = train['Transported'].value_counts()
target_pct    = train['Transported'].value_counts(normalize=True) * 100

print('件数:')
print(target_counts)
print('\n割合(%):')
print(target_pct.round(1))

target_counts.plot(kind='bar', color=['steelblue', 'coral'])
plt.title('Transported の分布')
plt.xticks(rotation=0)
plt.ylabel('件数')
plt.tight_layout()
plt.show()

## 4. 数値変数の分布と Transported との関係

In [ ]:
num_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    for transported, group in train.groupby('Transported'):
        axes[i].hist(group[col].dropna(), bins=40, alpha=0.5, label=str(transported))
    axes[i].set_title(col)
    axes[i].legend(title='Transported')

plt.suptitle('数値変数の分布 (Transportedで色分け)', y=1.02)
plt.tight_layout()
plt.show()

### 仮説検証: 年齢と CryoSleep の関係
**仮説**: 若年層は CryoSleep しやすく、それが Transported に影響しているのでは？

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ① CryoSleep 別の年齢分布
for cryo, group in train.groupby('CryoSleep'):
    axes[0].hist(group['Age'].dropna(), bins=40, alpha=0.5, label=str(cryo))
axes[0].set_title('CryoSleep 別の年齢分布')
axes[0].set_xlabel('Age')
axes[0].legend(title='CryoSleep')

# ② 年齢帯別の CryoSleep 率
train['AgeGroup'] = pd.cut(
    train['Age'],
    bins=[0, 12, 18, 30, 50, 80],
    labels=['子供', '10代', '20代', '30-50代', '50代以上']
)
cryo_rate = train.groupby('AgeGroup', observed=True)['CryoSleep'].apply(
    lambda x: x.astype(str).map({'True': 1, 'False': 0}).mean()
) * 100
cryo_rate.plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('年齢帯別の CryoSleep 率(%)')
axes[1].set_ylabel('CryoSleep 率(%)')
axes[1].axhline(50, color='red', linestyle='--')
axes[1].tick_params(axis='x', rotation=30)

# ③ 年齢帯 × CryoSleep 別の Transported 率
for cryo, group in train.groupby('CryoSleep'):
    rate = group.groupby('AgeGroup', observed=True)['Transported'].mean() * 100
    axes[2].plot(rate.index.astype(str), rate.values, marker='o', label=str(cryo))
axes[2].set_title('年齢帯 × CryoSleep 別の Transported 率(%)')
axes[2].set_ylabel('Transported 率(%)')
axes[2].legend(title='CryoSleep')
axes[2].axhline(50, color='red', linestyle='--')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 5. カテゴリ変数と Transported の関係

In [ ]:
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(cat_cols):
    ct = train.groupby(col)['Transported'].mean() * 100
    ct.plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(f'{col} 別の Transported 率(%)')
    axes[i].set_ylabel('Transported 率(%)')
    axes[i].set_ylim(0, 100)
    axes[i].axhline(50, color='red', linestyle='--', linewidth=0.8)
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 6. 相関分析

In [ ]:
corr_df = train[num_cols + ['Transported']].copy()
corr_df['Transported'] = corr_df['Transported'].astype(int)

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_df.corr(),
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True
)
plt.title('相関係数ヒートマップ')
plt.tight_layout()
plt.show()

### 仮説検証: 支出カラムを2グループに分けるべきか？
**仮説**: RoomService・Spa・VRDeck は Transported と負の相関が強く、FoodCourt・ShoppingMall は弱い → 合計より分けた方が有効では？

In [ ]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
trans = train['Transported'].astype(int)

# 各支出カラムと Transported の相関係数
corr_with_target = pd.Series(
    {col: train[col].corr(trans) for col in spend_cols}
).sort_values()

print('各支出カラムと Transported の相関係数:')
print(corr_with_target.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 相関係数の棒グラフ
colors = ['coral' if v < -0.05 else 'steelblue' for v in corr_with_target]
corr_with_target.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('各支出カラムと Transported の相関係数')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].tick_params(axis='x', rotation=30)

# 2グループに分けた合計の比較
spend_rsv = train[['RoomService', 'Spa', 'VRDeck']].sum(axis=1)
spend_fc  = train[['FoodCourt', 'ShoppingMall']].sum(axis=1)
total     = train[spend_cols].sum(axis=1)

comparison = pd.Series({
    'Spend_RSV\n(Room+Spa+VR)': spend_rsv.corr(trans),
    'Spend_FC\n(Food+Shop)':    spend_fc.corr(trans),
    'TotalSpend\n(全合計)':      total.corr(trans)
})
comparison.plot(kind='bar', ax=axes[1], color=['coral', 'steelblue', 'gray'])
axes[1].set_title('グループ別合計 vs Transported の相関係数')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(f'\nSpend_RSV と Transported の相関: {spend_rsv.corr(trans):.3f}')
print(f'Spend_FC  と Transported の相関: {spend_fc.corr(trans):.3f}')
print(f'TotalSpend と Transported の相関: {total.corr(trans):.3f}')

## 7. 特徴量エンジニアリング

In [ ]:
def add_features(df):
    df = df.copy()

    # Cabin を Deck / Side に分割
    cabin_split = df['Cabin'].str.split('/', expand=True)
    df['Deck'] = cabin_split[0]
    df['Side'] = cabin_split[2]

    # PassengerId からグループ情報を抽出
    df['GroupId']   = df['PassengerId'].str.split('_').str[0]
    df['GroupSize'] = df.groupby('GroupId')['GroupId'].transform('count')
    df['IsAlone']   = (df['GroupSize'] == 1).astype(int)

    # 支出合計
    spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
    df['TotalSpend'] = df[spend_cols].sum(axis=1)
    df['HasSpend']   = (df['TotalSpend'] > 0).astype(int)

    # 仮説検証で有効だったグループ別支出
    df['Spend_RSV'] = df[['RoomService', 'Spa', 'VRDeck']].sum(axis=1)
    df['Spend_FC']  = df[['FoodCourt', 'ShoppingMall']].sum(axis=1)

    return df

train = add_features(train)
test  = add_features(test)

print('追加した特徴量: Deck, Side, GroupSize, IsAlone, TotalSpend, HasSpend, Spend_RSV, Spend_FC')
train[['Deck', 'Side', 'GroupSize', 'IsAlone', 'TotalSpend', 'Spend_RSV', 'Spend_FC']].head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Deck別
train.groupby('Deck')['Transported'].mean().sort_values().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Deck 別 Transported 率')
axes[0].axhline(0.5, color='red', linestyle='--')

# Side別
train.groupby('Side')['Transported'].mean().plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Side 別 Transported 率')
axes[1].axhline(0.5, color='red', linestyle='--')

# TotalSpend vs Transported
for t, g in train.groupby('Transported'):
    axes[2].hist(np.log1p(g['TotalSpend']), bins=40, alpha=0.5, label=str(t))
axes[2].set_title('log(TotalSpend+1) の分布')
axes[2].legend(title='Transported')

plt.tight_layout()
plt.show()

## 8. 仮説検証: Age=0 の扱い方で精度は変わるか？

4パターンを比較する
- A: 何もしない（ベースライン）
- B: Age=0 を NaN に置換
- C: Age_is_zero フラグを追加
- D: B + C（両方）

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

BASE_FEATURES = [
    'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
    'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
    'Deck', 'Side', 'GroupSize', 'IsAlone',
    'TotalSpend', 'HasSpend', 'Spend_RSV', 'Spend_FC'
]
CAT_COLS = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']

def encode_and_score(X, y, label):
    X = X.copy()
    for col in CAT_COLS:
        if col in X.columns:
            le = LabelEncoder()
            X[col] = le.fit_transform(X[col].astype(str))
    model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1)
    scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
    print(f'{label}: {scores.mean():.4f} ± {scores.std():.4f}')
    return scores.mean()

y = train['Transported'].astype(int)
results = {}

# A: ベースライン（何もしない）
X_a = train[BASE_FEATURES].copy()
results['A: ベースライン'] = encode_and_score(X_a, y, 'A: ベースライン')

# B: Age=0 を NaN に置換
X_b = train[BASE_FEATURES].copy()
X_b['Age'] = X_b['Age'].replace(0, float('nan'))
results['B: Age=0→NaN'] = encode_and_score(X_b, y, 'B: Age=0→NaN')

# C: Age_is_zero フラグを追加
X_c = train[BASE_FEATURES].copy()
X_c['Age_is_zero'] = (X_c['Age'] == 0).astype(int)
results['C: フラグ追加'] = encode_and_score(X_c, y, 'C: フラグ追加')

# D: B + C（両方）
X_d = train[BASE_FEATURES].copy()
X_d['Age_is_zero'] = (X_d['Age'] == 0).astype(int)
X_d['Age'] = X_d['Age'].replace(0, float('nan'))
results['D: NaN+フラグ'] = encode_and_score(X_d, y, 'D: NaN+フラグ')

# 結果の可視化
print()
result_series = pd.Series(results)
best = result_series.idxmax()
print(f'最高精度: {best} ({result_series[best]:.4f})')

result_series.plot(kind='bar', color=['gray', 'steelblue', 'coral', 'green'], figsize=(8, 4))
plt.title('Age=0 の処理方法による精度比較')
plt.ylabel('CV Accuracy')
plt.ylim(result_series.min() - 0.005, result_series.max() + 0.005)
plt.xticks(rotation=15)
plt.axhline(results['A: ベースライン'], color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## 9. 仮説検証: 支出カラムを分けると精度は上がるか？

3パターンを比較する
- A: TotalSpend のみ（合計）
- B: Spend_RSV + Spend_FC に分割
- C: 5カラムそれぞれ個別に使用

In [ ]:
BASE = ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
        'Deck', 'Side', 'GroupSize', 'IsAlone', 'HasSpend']
CAT  = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']

# Age=0 を NaN に置換（前の検証で有効だったため採用）
train_v = train.copy()
train_v['Age'] = train_v['Age'].replace(0, float('nan'))
y = train_v['Transported'].astype(int)

results = {}

# A: TotalSpend のみ
X_a = train_v[BASE + ['TotalSpend']].copy()
results['A: TotalSpend'] = encode_and_score(X_a, y, 'A: TotalSpend')

# B: Spend_RSV + Spend_FC に分割
X_b = train_v[BASE + ['Spend_RSV', 'Spend_FC']].copy()
results['B: RSV+FC分割'] = encode_and_score(X_b, y, 'B: RSV+FC分割')

# C: 5カラム個別
X_c = train_v[BASE + ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].copy()
results['C: 5カラム個別'] = encode_and_score(X_c, y, 'C: 5カラム個別')

# 結果の可視化
print()
result_series = pd.Series(results)
best = result_series.idxmax()
print(f'最高精度: {best} ({result_series[best]:.4f})')

result_series.plot(kind='bar', color=['gray', 'steelblue', 'coral'], figsize=(8, 4))
plt.title('支出カラムの集約方法による精度比較')
plt.ylabel('CV Accuracy')
plt.ylim(result_series.min() - 0.005, result_series.max() + 0.005)
plt.xticks(rotation=15)
plt.axhline(results['A: TotalSpend'], color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

## 10. ベースラインモデル (LightGBM)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

FEATURES = [
    'HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP',
    'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
    'Deck', 'Side', 'GroupSize', 'IsAlone',
    'TotalSpend', 'HasSpend', 'Spend_RSV', 'Spend_FC'
]
TARGET = 'Transported'

X = train[FEATURES].copy()
y = train[TARGET].astype(int)

cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Deck', 'Side']
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1)

scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f'CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')
print(f'各fold: {scores.round(4)}')

## 11. 特徴量重要度

In [ ]:
importance_df = pd.DataFrame({
    'feature': FINAL_FEATURES,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=True)

importance_df.plot(kind='barh', x='feature', y='importance',
                   figsize=(8, 7), legend=False, color='steelblue')
plt.title('特徴量重要度 (LightGBM — チューニング済み)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 12. ハイパーパラメータチューニング (Optuna)

### 目的
過学習を抑えつつ CV AUC を最大化する

### 戦略
- **評価指標**: AUC（確率スコアを使うため、Accuracy より安定）
- **探索手法**: TPE (Tree-structured Parzen Estimator) — ランダム探索より効率的
- **Early Stopping**: 各 fold で検証データを使い、過学習し始めたら学習を止める
- **n_estimators**: 1000 に固定し、早期停止で最適ステップ数を自動決定

### 探索パラメータと探索範囲の理由

| パラメータ | 範囲 | 理由 |
|-----------|------|------|
| `learning_rate` | 0.01〜0.3 (log) | 小さいほど丁寧に学習。log スケールで広く探索 |
| `num_leaves` | 20〜300 | 葉の数 = モデルの複雑さ。デフォルト31を中心に広めに探索 |
| `max_depth` | 3〜12 | 木の深さ制限。深すぎると過学習 |
| `min_data_in_leaf` | 10〜100 | 1葉の最小サンプル数。大きいほど汎化しやすい |
| `feature_fraction` | 0.5〜1.0 | 1ツリーあたり使う特徴量の割合。ランダム性で過学習防止 |
| `bagging_fraction` | 0.5〜1.0 | 1ツリーあたり使うデータの割合。ランダム性で過学習防止 |
| `lambda_l1` | 0〜10 | L1正則化。重要でない特徴量の重みを0にする |
| `lambda_l2` | 0〜10 | L2正則化。重みを全体的に小さくする |

In [ ]:
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import numpy as np

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── 最終特徴量セット（仮説検証で確定） ──────────────────────────────
FINAL_FEATURES = [
    'HomePlanet', 'CryoSleep', 'Age', 'VIP', 'Deck', 'Side',
    'GroupSize', 'IsAlone', 'HasSpend', 'Spend_RSV', 'Spend_FC', 'Is55Cancri'
]
CAT_COLS_FINAL = ['HomePlanet', 'CryoSleep', 'VIP', 'Deck', 'Side']

# ── データ準備 ────────────────────────────────────────────────────────
train_opt = train.copy()
train_opt['Age']       = train_opt['Age'].replace(0, float('nan'))   # Age=0 → NaN
train_opt['Is55Cancri'] = (train_opt['Destination'] == '55 Cancri e').astype(int)

X_opt = train_opt[FINAL_FEATURES].copy()
y_opt = train_opt['Transported'].astype(int)

# カテゴリ変数をラベルエンコード
le_dict = {}
for col in CAT_COLS_FINAL:
    le = LabelEncoder()
    X_opt[col] = le.fit_transform(X_opt[col].astype(str))
    le_dict[col] = le

print('チューニング用データ準備完了')
print(f'特徴量数: {len(FINAL_FEATURES)}, サンプル数: {len(X_opt)}')

In [ ]:
def objective(trial):
    """Optuna の目的関数: 5-fold CV の平均 AUC を最大化"""
    params = {
        # 学習率: log スケールで探索（小さい値ほど細かく学習）
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        # 葉の数: モデルの複雑さを制御（大きいほど複雑）
        'num_leaves':       trial.suggest_int('num_leaves', 20, 300),
        # 木の最大深さ: 深すぎると過学習
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        # 1葉に必要な最小サンプル数: 大きいほど汎化しやすい
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        # 1ツリーで使う特徴量の割合: ランダム性で過学習を抑える
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        # 1ツリーで使うデータの割合: ランダム性で過学習を抑える
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':     1,           # bagging_fraction を有効化するために必要
        # L1 正則化: 特徴量の重みをスパース（0）にする
        'lambda_l1':        trial.suggest_float('lambda_l1', 0.0, 10.0),
        # L2 正則化: 重みを全体的に小さく抑える
        'lambda_l2':        trial.suggest_float('lambda_l2', 0.0, 10.0),
        # n_estimators は大きく固定し、early stopping で実際の本数を決める
        'n_estimators':     1000,
        'random_state':     42,
        'verbose':          -1,
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_aucs = []

    for tr_idx, val_idx in skf.split(X_opt, y_opt):
        X_tr,  X_val  = X_opt.iloc[tr_idx],  X_opt.iloc[val_idx]
        y_tr,  y_val  = y_opt.iloc[tr_idx],  y_opt.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            callbacks=[
                # 50ステップ改善がなければ学習を打ち切り（過学習防止）
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=-1),   # ログを非表示
            ],
        )

        pred = model.predict_proba(X_val)[:, 1]
        fold_aucs.append(roc_auc_score(y_val, pred))

    return np.mean(fold_aucs)

# ── Optuna スタディの実行（n_trials=100 で約3〜8分） ─────────────────
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f'\nBest CV AUC : {study.best_value:.4f}')
print('Best params :')
for k, v in study.best_params.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

In [ ]:
# ── 探索結果の可視化 ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ① AUC の推移
trials_df = study.trials_dataframe()
axes[0].plot(trials_df['number'], trials_df['value'],
             alpha=0.3, color='steelblue', label='各 trial')
axes[0].plot(trials_df['number'], trials_df['value'].cummax(),
             color='red', linewidth=2, label='最高 AUC')
axes[0].set_xlabel('Trial 番号')
axes[0].set_ylabel('CV AUC')
axes[0].set_title('Optuna 探索の推移')
axes[0].legend()

# ② パラメータ重要度（どのパラメータが AUC に影響したか）
importance = optuna.importance.get_param_importances(study)
imp_series = pd.Series(importance).sort_values()
imp_series.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('パラメータの重要度')
axes[1].set_xlabel('重要度')

plt.tight_layout()
plt.show()

print('>>> パラメータ重要度の読み方')
print('値が大きい = そのパラメータを変えると AUC が大きく変わった')

In [ ]:
# ── 最適パラメータで最終 CV 評価 ──────────────────────────────────────
best_params = study.best_params.copy()
best_params.update({
    'bagging_freq':  1,       # bagging_fraction のために必要
    'n_estimators':  1000,    # early stopping で自動決定
    'random_state':  42,
    'verbose':       -1,
})

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
final_aucs = []
final_accs = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_opt, y_opt)):
    X_tr,  X_val  = X_opt.iloc[tr_idx],  X_opt.iloc[val_idx]
    y_tr,  y_val  = y_opt.iloc[tr_idx],  y_opt.iloc[val_idx]

    m = lgb.LGBMClassifier(**best_params)
    m.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=-1),
        ],
    )

    pred_p = m.predict_proba(X_val)[:, 1]
    pred_l = (pred_p >= 0.5).astype(int)

    auc = roc_auc_score(y_val, pred_p)
    acc = (pred_l == y_val.values).mean()
    print(f'Fold {fold+1}: AUC={auc:.4f}  Accuracy={acc:.4f}  '
          f'best_iteration={m.best_iteration_}')
    final_aucs.append(auc)
    final_accs.append(acc)

print(f'\n最終 CV AUC      : {np.mean(final_aucs):.4f} ± {np.std(final_aucs):.4f}')
print(f'最終 CV Accuracy : {np.mean(final_accs):.4f} ± {np.std(final_accs):.4f}')

## 13. 提出ファイルの作成

### 最終モデルの学習方針
- 全 train データで1つのモデルを学習する
- early stopping には検証データが必要なため、`n_estimators` は各 fold の `best_iteration` の平均値 × 1.1 を使う
- テストデータのエンコードは train で学習した LabelEncoder をそのまま使う（データリークを防ぐため）

In [ ]:
# ── テストデータの準備 ────────────────────────────────────────────────
test_opt = test.copy()
test_opt['Age']        = test_opt['Age'].replace(0, float('nan'))
test_opt['Is55Cancri'] = (test_opt['Destination'] == '55 Cancri e').astype(int)

X_test_opt = test_opt[FINAL_FEATURES].copy()

# train で学習した LabelEncoder を使ってエンコード（データリーク防止）
for col in CAT_COLS_FINAL:
    X_test_opt[col] = le_dict[col].transform(X_test_opt[col].astype(str))

# ── best_iteration の平均から最終 n_estimators を決定 ────────────────
best_iters = [455, 539, 547, 583, 373]   # 各 fold の best_iteration
final_n_estimators = int(np.mean(best_iters) * 1.1)  # 平均の 1.1 倍（全データ分の余裕）
print(f'使用する n_estimators: {final_n_estimators}')

# ── 全 train データで最終モデルを学習 ────────────────────────────────
final_params = best_params.copy()
final_params['n_estimators'] = final_n_estimators

final_model = lgb.LGBMClassifier(**final_params)
final_model.fit(X_opt, y_opt)

print('最終モデルの学習完了')

# ── 予測・提出ファイル作成 ────────────────────────────────────────────
preds = final_model.predict(X_test_opt)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': preds.astype(bool)
})
submission.to_csv('spaceship-titanic/submission.csv', index=False)

print('提出ファイル作成完了: spaceship-titanic/submission.csv')
print(f'Transported=True  : {submission["Transported"].sum()} 件')
print(f'Transported=False : {(~submission["Transported"]).sum()} 件')
submission.head()